In [1]:
from pathlib import Path
from typing import Dict

import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import torch
import pickle
from torch.utils.data import DataLoader

from neuralhydrology.datasetzoo import get_dataset, lamah
from neuralhydrology.datautils.utils import load_scaler
from neuralhydrology.modelzoo.ealstm import EALSTM
from neuralhydrology.modelzoo.customlstm import CustomLSTM
from neuralhydrology.nh_run import eval_run
from neuralhydrology.utils.config import Config

In [ ]:
with open("/home/wuhlmann/BA/test_runs/runs/128_q_filter_1311_143334/train_data/train_data.p", "rb") as f: 
    ds = pickle.load(f)

In [ ]:
ds["coords"]["basin"]["data"]

In [ ]:
nse_dict = {}

In [ ]:
run_dir_path = Path("/home/wuhlmann/BA/test_runs/runs/256_q_filter_1711_155953")

In [ ]:
attributes_shape_path = Path("/home/wuhlmann/BA/data/raw_data/2_LamaH-CE_daily/B_basins_intermediate_all/3_shapefiles/Basins_B.shp")
basin_stca_gdf = gpd.read_file(attributes_shape_path)
basin_stca_gdf.set_index("ID", inplace=True)

for phase in ["train", "validation", "test"]:

    metrics_path = run_dir_path / f"{phase}/model_epoch015/{phase}_metrics.csv" 

    metrics = pd.read_table(metrics_path, header=0, sep=",")

    nse_dict[phase] = metrics
        


In [5]:
cfg = Config(Path("/home/wuhlmann/BA/test_runs/runs/256_q_filter_1711_155954/config.yml"))
ds = get_dataset(cfg=cfg, is_train=True, period="train")

100%|██████████| 669/669 [00:57<00:00, 11.71it/s]


In [8]:
ds._attributes["133"]

tensor([-0.7942,  0.0039, -1.3066,  0.0637, -0.5666, -0.2168,  0.3019,  1.1379,
        -0.5905,  1.0345,  0.9317, -0.1625,  0.9964,  0.9894,  0.1754,  0.5155,
        -1.4497, -0.2407, -1.3720, -1.4788,  0.5139, -0.7998,  0.1594,  2.2068,
        -0.2727, -0.4590,  0.8679,  0.9933, -0.9934,  0.3715, -0.5313])

In [ ]:
run_dir_path = Path("/home/wuhlmann/BA/test_runs/runs/256_q_filter_1711_155954")
with open(run_dir_path / "test/model_epoch015/test_results.p", "rb") as f: 
    raw_test_results = pickle.load(f)

In [ ]:
test_metrics = pd.read_table(run_dir_path / "test/model_epoch015/test_metrics.csv", sep=",")
test_metrics["NSE"].median()

In [ ]:
phase = "train"

fig, ax = plt.subplots(figsize=(8, 8))

df = nse_dict[phase]

print(f"{phase} NSE median is {df['NSE'].median()}")

sub_0_basins = df[df["NSE"] < 0]

test_basins = basin_stca_gdf.loc[df["basin"].values]
test_basins.plot(ax=ax, color="lightblue", edgecolor="gray")
highlight = basin_stca_gdf.loc[sub_0_basins["basin"].values]
highlight.plot(ax=ax, color="red", edgecolor="black")

In [ ]:
sub_0_basins

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

sub_0_basins = []

for phase, df in nse_dict.items():

    print(f"{phase} NSE median {df['NSE'].median():.3f} | mean {df['NSE'].mean():.3f}")

    sub_0_basins.extend(df[df["NSE"]<0]["basin"].values)

basin_stca_gdf.plot(ax=ax, color="lightblue", edgecolor="gray")
highlight = basin_stca_gdf.loc[sub_0_basins]
highlight.plot(ax=ax, color="red", edgecolor="black")

